# 

In [7]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from dataset_ood_download import get_data_list
import selfies as sf

In [8]:

data = load_dataset("OpenMol/RCR_RP_57K_SMILES-MMChat")
task_name = "presto-reagent_prediction"
instruction_templates = instructions_smol.reagent_prediction

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(data['test'])))
for i in iter_bar:
    smiles = data['test'][i]['molecules']['smiles']
    label = data['test'][i]['ground_truth']
    if isinstance(smiles, list) and len(smiles) > 1:
        mol = [Chem.MolFromSmiles(s) for s in smiles]
    else:
        mol = Chem.MolFromSmiles(smiles[0])
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue
    
list_data = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
)
print(len(list_data), len(omitted_idx))
rp_dataset = datasets.Dataset.from_list(list_data)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

rp_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}_0405")
rp_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}_0405")
rp_dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}_0405")

print(rp_dataset[0])

100%|██████████| 6377/6377 [00:30<00:00, 211.77it/s]


6377 1


Saving the dataset (1/1 shards): 100%|██████████| 6377/6377 [00:00<00:00, 10190.92 examples/s]

{'task': 'presto-reagent_prediction', 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0]

In [9]:
data = load_dataset("OpenMol/MolInst_FS_125K_Scaffold_SMILES-MMChat")
task_name = "presto-forward_reaction_prediction"
instruction_templates = instructions_smol.forward_reaction_prediction

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(data['test'])))
for i in iter_bar:
    smiles = data['test'][i]['molecules']['smiles']
    label = data['test'][i]['ground_truth']
    if isinstance(smiles, list):
        combined_smiles = '.'.join(smiles)
        mol = Chem.MolFromSmiles(combined_smiles)
    else:
        raise ValueError("smiles should be a list")
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue

list_data = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
)
print(len(list_data), len(omitted_idx))
dataset = datasets.Dataset.from_list(list_data)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}_0405")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}_0405")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}_0405")

print(dataset[0])

 10%|▉         | 99/1004 [00:00<00:01, 506.49it/s][05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
[05:43:22] WARNING: not removing hydrogen atom without neighbors
 15%|█▍        | 150/1004 [00:00<00:02, 424.62it/s][05:43:22] WARNING: not removing hydrogen atom without

1004 0


Saving the dataset (1/1 shards): 100%|██████████| 1004/1004 [00:00<00:00, 5286.95 examples/s]


{'task': 'presto-forward_reaction_prediction', 'x': [[7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [5, 0, 3, 5, 0, 0, 1, 0, 0], [7, 0, 2, 5, 1, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [8, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 3, 5, 1, 0, 1, 0, 0], [7, 0, 1, 5, 0, 0, 1, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [16, 0, 1, 5, 0, 0, 2, 0, 0]], 'edge_index': [[0, 1, 1, 2, 1, 3, 3, 4, 3, 5, 6, 7, 7, 8, 7, 9, 9, 10, 10, 11, 11, 12, 11, 13, 11, 14, 15, 16, 16, 17, 16, 18, 18, 19, 20, 21, 21, 22], [1, 0, 2, 1, 3, 1, 4, 3, 5, 3, 7, 6, 8, 7, 9, 7, 10, 9, 11, 10, 12, 11, 13, 11, 14, 11, 16, 15, 17, 16, 18, 16, 19, 18, 21,

In [10]:
data = load_dataset("OpenMol/MolInst_RS_125K_Scaffold_SMILES-MMChat")
task_name = "presto-retrosynthesis"
instruction_templates = instructions_smol.retrosynthesis

list_mol = []
list_label = []
omitted_idx = []    
from tqdm import tqdm
iter_bar = tqdm(range(len(data['test'])))
for i in iter_bar:
    smiles = data['test'][i]['molecules']['smiles']
    label = data['test'][i]['ground_truth']
    if isinstance(smiles, list):
        combined_smiles = '.'.join(smiles)
        mol = Chem.MolFromSmiles(combined_smiles)
    else:
        raise ValueError("smiles should be a list")
    try:
        label = sf.encoder(label)
        list_label.append(label)
        list_mol.append(mol)
    except:
        omitted_idx.append(i)
        continue

list_data = get_data_list(
    list_mol=list_mol,
    list_label=list_label,
    task=task_name,
    instruction_templates=instruction_templates,
)
print(len(list_data), len(omitted_idx))
dataset = datasets.Dataset.from_list(list_data)

llm_model = "mistralai/Mistral-7B-Instruct-v0.3"
mol_representation = "string+graph"
num_query_token = 32
base_model = llm_model.replace("/", "-")
tags = [base_model, mol_representation]
if "graph" in mol_representation:
    tags += [f"q{num_query_token}"]

processed_file_name = "_".join(tags)
raw_data_root = "/data/data/Mol-LLM-v7.1"

dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_train_{task_name}_0405")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_test_{task_name}_0405")
dataset.save_to_disk(f"{raw_data_root}/{processed_file_name}_validation_{task_name}_0405")

print(dataset[0])

100%|██████████| 1000/1000 [00:03<00:00, 322.71it/s]


1000 0


Saving the dataset (1/1 shards): 100%|██████████| 1000/1000 [00:00<00:00, 6029.46 examples/s]

{'task': 'presto-retrosynthesis', 'x': [[5, 0, 4, 5, 3, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 4, 5, 2, 0, 2, 0, 0], [7, 0, 2, 5, 0, 0, 2, 0, 0], [5, 0, 4, 5, 3, 0, 2, 0, 0], [6, 0, 2, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [6, 0, 3, 5, 2, 0, 1, 0, 0], [6, 0, 2, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [7, 0, 2, 5, 0, 0, 1, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [5, 0, 4, 5, 2, 0, 2, 0, 0], [6, 0, 3, 5, 0, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 4, 5, 2, 0, 2, 0, 1], [5, 0, 3, 5, 1, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1], [5, 0, 3, 5, 0, 0, 1, 1, 1]], 'edge_index': [[0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 7, 4, 8, 8, 9, 9, 10, 10, 11, 10, 12, 12, 13, 13, 14, 14, 15, 1